# 05. Modeling and Evaluation

**Objective:** Compare exactly four models: **A — Naive baseline**, **B — Historical LightGBM**,
**C — Graph-enhanced LightGBM**, and **D — Graph Convolutional Network (GCN)**.
Models B and C retain the existing validation MAE hyperparameter search; the GCN
retains validation MAE early stopping. **Precision@20 and Capture@20 are the primary
hotspot decision metrics**; MAE and R² provide supplementary count-fit context.

Run from the `notebooks` directory after notebooks 03 and 04. The existing environment
needs GeoPandas, LightGBM, scikit-learn, joblib, PyTorch, and PyTorch Geometric.
No checkpoint or prediction from an earlier feature configuration is loaded for training.

## 1. Load data and define chronological splits

Load the verified modeling dataset, seed the existing models, and preserve the
2020–2023 training, 2024 validation, and exploratory 2025 evaluation periods.

In [1]:
import copy
import os
import platform
from pathlib import Path

# Avoid the local LightGBM/PyTorch native-thread crash when reloading models.
os.environ["OMP_NUM_THREADS"] = "1"

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

/Users/andrew/Documents/Projects/portfolio/3.Collisionprediction/.col/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/andrew/Documents/Projects/portfolio/3.Collisionprediction/.col/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
model_data = pd.read_csv("../data/processed/model_data.csv")
model_data["quarter"] = pd.PeriodIndex(model_data["quarter"], freq="Q")
model_data = model_data.sort_values(["cell_id", "quarter"]).reset_index(drop=True)

assert model_data.shape == (8112, 13)
assert not model_data.duplicated(["cell_id", "quarter"]).any()
assert model_data["cell_id"].nunique() == 338

train = model_data[model_data["quarter"].between("2020Q1", "2023Q4")].copy()
val = model_data[model_data["quarter"].between("2024Q1", "2024Q4")].copy()
test = model_data[model_data["quarter"].between("2025Q1", "2025Q4")].copy()

for frame, start, end in [
    (train, "2020Q1", "2023Q4"),
    (val, "2024Q1", "2024Q4"),
    (test, "2025Q1", "2025Q4"),
]:
    expected = pd.period_range(start, end, freq="Q")
    assert sorted(frame["quarter"].unique()) == list(expected)
    assert frame.groupby("quarter").size().eq(338).all()
assert len(train) + len(val) + len(test) == len(model_data)
assert train["quarter"].max() < val["quarter"].min() < test["quarter"].min()
print("Train / validation / test rows:", len(train), len(val), len(test))

Train / validation / test rows: 5408 1352 1352


## 2. Model A — Naive historical baseline

Use the previous quarter's collision count (`lag_1`) directly, without fitting a model.
All models forecast one quarter ahead using history available before each quarter.

In [3]:
val["baseline_pred"] = val["lag_1"]

## 3. Models B and C — LightGBM

Model B uses four historical features. Model C adds four road-network features.
Both use the original hyperparameter grid, fitted on 2020–2023 and selected on
2024 validation MAE. The comparison below reports one selected model per feature set.

In [4]:
historical_cols = [
    "lag_1",
    "lag_4",
    "rolling_mean_4",
    "quarter_of_year"
]
graph_cols = [
    "road_length_m",
    "major_road_share",
    "intersection_density",
    "mean_streets_per_node"
]
model_c_features = historical_cols + graph_cols

assert len(historical_cols) == 4
assert len(model_c_features) == 8
assert np.isfinite(model_data[model_c_features].to_numpy()).all()

In [5]:
param_grid = {
    "n_estimators": [100, 200],
    "num_leaves": [7, 15, 31],
    "min_child_samples": [20, 50]
}

def tune_model(features):
    best_mae = float("inf")
    best_model = None
    best_params = None

    for params in ParameterGrid(param_grid):
        model = LGBMRegressor(
            **params,
            learning_rate=0.05,
            random_state=SEED,
            verbosity=-1,
            n_jobs=1
        )

        model.fit(train[features], train["collisions"])

        predictions = np.maximum(
            model.predict(val[features]), 0
        )

        mae = mean_absolute_error(
            val["collisions"], predictions
        )

        if mae < best_mae:
            best_mae = mae
            best_model = model
            best_params = params

    return best_model, best_mae, best_params


# Tune both using the same search space
best_model_b, best_mae_b, params_b = tune_model(historical_cols)
best_model_c, best_mae_c, params_c = tune_model(
    model_c_features
)

print("Model B:", best_mae_b, params_b)
print("Model C:", best_mae_c, params_c)

Model B: 2.3345403907682414 {'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 7}
Model C: 2.1496895383026557 {'min_child_samples': 50, 'n_estimators': 200, 'num_leaves': 31}


In [6]:
val["model_b_pred"] = np.maximum(best_model_b.predict(val[historical_cols]), 0)
val["model_c_pred"] = np.maximum(best_model_c.predict(val[model_c_features]), 0)

## 4. Hotspot evaluation

For each quarter, rank cells by predicted collisions and compare the top 20 with
the observed top 20. Ties use `cell_id` for reproducibility. **Precision@20** is the
fraction of selected cells in the observed top 20; **Capture@20** is the fraction
of all collisions occurring in selected cells. Average both metrics across quarters.

In [7]:
def evaluate_model(df, prediction_col, k=20):
    assert k == 20, "This experiment reports Precision@20 and Capture@20."
    assert not df.empty
    assert not df.duplicated(["cell_id", "quarter"]).any()
    assert np.isfinite(df[["collisions", prediction_col]].to_numpy()).all()
    results = []

    for quarter, group in df.groupby("quarter", sort=True):
        assert len(group) >= k
        predicted_top = group.sort_values(
            [prediction_col, "cell_id"], ascending=[False, True]
        ).head(k)
        actual_top = group.sort_values(
            ["collisions", "cell_id"], ascending=[False, True]
        ).head(k)
        overlap = len(set(predicted_top["cell_id"]) & set(actual_top["cell_id"]))
        total = group["collisions"].sum()
        results.append({
            "quarter": str(quarter),
            "precision_at_20": overlap / k,
            "capture_at_20": predicted_top["collisions"].sum() / total if total else 0.0
        })

    quarterly_results = pd.DataFrame(results)
    return {
        "Precision@20": quarterly_results["precision_at_20"].mean(),
        "Capture@20": quarterly_results["capture_at_20"].mean(),
        "MAE": mean_absolute_error(df["collisions"], df[prediction_col]),
        "R²": r2_score(df["collisions"], df[prediction_col])
    }

## 5. Model D — Graph Convolutional Network

The same eight features as Model C describe 338 grid-cell nodes. Edges connect
cells sharing a boundary segment; corner-only contacts are excluded. The existing
two-layer GCN (32 and 16 hidden units), Adam optimizer, MSE training loss, validation
MAE checkpoint selection, and early stopping are retained. Scaling uses training
rows only. Every run initializes a new eight-input model and retrains it.

In [8]:


print("Architecture:", platform.machine())
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Selected device:", device)

Architecture: arm64
PyTorch: 2.15.0
PyTorch Geometric: 2.8.1
MPS available: True
Selected device: mps


In [9]:
grid = gpd.read_file("../data/processed/manhattan_grid.gpkg").to_crs("EPSG:32618")
grid = grid[["cell_id", "geometry"]].sort_values("cell_id").reset_index(drop=True)
assert len(grid) == 338 and grid["cell_id"].is_unique
cell_ids = grid["cell_id"].tolist()
id_to_idx = {cell_id: i for i, cell_id in enumerate(cell_ids)}
assert set(cell_ids) == set(model_data["cell_id"])

neighbors = gpd.sjoin(grid, grid, how="inner", predicate="touches")
geometry = grid.set_index("cell_id").geometry
# Keep shared-edge neighbors, excluding corner-only contacts.
edge_pairs = sorted({
    (a, b)
    for a, b in zip(neighbors["cell_id_left"], neighbors["cell_id_right"])
    if a < b
    and geometry.loc[a].boundary.intersection(geometry.loc[b].boundary).length > 1
})
# PyTorch Geometric needs both directions, mapped from cell IDs to row indices.
directed_edges = edge_pairs + [(b, a) for a, b in edge_pairs]
edge_index = torch.tensor([
    [id_to_idx[a] for a, b in directed_edges],
    [id_to_idx[b] for a, b in directed_edges]
], dtype=torch.long)
assert edge_index.shape == (2, 2 * len(edge_pairs))
assert edge_index.numel() > 0
assert edge_index.min().item() >= 0 and edge_index.max().item() < len(cell_ids)
assert set(map(tuple, edge_index.T.tolist())) == {
    (id_to_idx[a], id_to_idx[b]) for a, b in directed_edges
}
assert set(directed_edges) == {(b, a) for a, b in directed_edges}
connected_cells = {cell_id for pair in edge_pairs for cell_id in pair}
print("Nodes:", len(cell_ids), "| Shared edges:", len(edge_pairs))
print("Isolated cells:", len(set(cell_ids) - connected_cells))
print("Edge index shape:", edge_index.shape)

Nodes: 338 | Shared edges: 592
Isolated cells: 0
Edge index shape: torch.Size([2, 1184])


In [10]:
gnn_features = historical_cols + graph_cols

assert len(gnn_features) == 8

# Fit scaling ONLY on training data
scaler = StandardScaler()
scaler.fit(train[gnn_features].to_numpy(dtype=np.float32))

def make_graph_batch(df):
    graphs = []

    for quarter in sorted(df["quarter"].unique()):
        q_rows = df[df["quarter"] == quarter]
        assert q_rows["cell_id"].is_unique
        assert set(q_rows["cell_id"]) == set(cell_ids)
        q_data = (
            df[df["quarter"] == quarter]
            .set_index("cell_id")
            .reindex(cell_ids)
        )

        assert len(q_data) == len(cell_ids)
        assert q_data[gnn_features + ["collisions"]].notna().all().all()

        x = scaler.transform(
            q_data[gnn_features].to_numpy(dtype=np.float32)
        )

        y = q_data["collisions"].to_numpy(dtype=np.float32)

        graphs.append(Data(
            x=torch.tensor(x, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor(y, dtype=torch.float32)
        ))

    return Batch.from_data_list(graphs)


train_batch = make_graph_batch(train)
val_batch = make_graph_batch(val)

assert train_batch.num_graphs == 16 and val_batch.num_graphs == 4
assert train_batch.x.shape == (5408, 8)
assert scaler.n_samples_seen_ == len(train)

print("Training graphs:", train_batch.num_graphs)
print("Validation graphs:", val_batch.num_graphs)
print("Training feature shape:", train_batch.x.shape)

Training graphs: 16
Validation graphs: 4
Training feature shape: torch.Size([5408, 8])


In [11]:
class RoadRiskGCN(nn.Module):
    def __init__(self, num_features):
        super().__init__()

        self.conv1 = GCNConv(num_features, 32)
        self.conv2 = GCNConv(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # Collision counts cannot be negative
        return F.softplus(self.output(x)).squeeze(-1)


torch.manual_seed(SEED)

gnn = RoadRiskGCN(len(gnn_features)).to(device)

train_batch = train_batch.to(device)
val_batch = val_batch.to(device)

optimizer = torch.optim.Adam(
    gnn.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

best_mae = float("inf")
best_state = None
best_epoch = 0
patience = 35

for epoch in range(1, 2001):
    gnn.train()
    optimizer.zero_grad()

    predictions = gnn(
        train_batch.x,
        train_batch.edge_index
    )

    loss = F.mse_loss(predictions, train_batch.y)
    loss.backward()
    optimizer.step()

    # Evaluate on 2024 validation data
    gnn.eval()

    with torch.no_grad():
        val_predictions = gnn(
            val_batch.x,
            val_batch.edge_index
        )

        val_mae = F.l1_loss(
            val_predictions,
            val_batch.y
        ).item()

    if val_mae < best_mae - 1e-4:
        best_mae = val_mae
        best_epoch = epoch
        best_state = copy.deepcopy(gnn.state_dict())

    if epoch % 25 == 0:
        print(
            f"Epoch {epoch} | "
            f"Train MSE: {loss.item():.3f} | "
            f"Validation MAE: {val_mae:.3f}"
        )

    if epoch - best_epoch >= patience:
        print("Early stopping")
        break

assert best_state is not None, "No finite validation checkpoint was selected."
gnn.load_state_dict(best_state)

print(f"Best epoch: {best_epoch}")
print(f"Best validation MAE: {best_mae:.3f}")

Epoch 25 | Train MSE: 53.818 | Validation MAE: 3.695
Epoch 50 | Train MSE: 34.580 | Validation MAE: 3.385
Epoch 75 | Train MSE: 30.920 | Validation MAE: 3.293
Epoch 100 | Train MSE: 29.277 | Validation MAE: 3.226
Epoch 125 | Train MSE: 28.092 | Validation MAE: 3.163
Epoch 150 | Train MSE: 27.173 | Validation MAE: 3.138
Epoch 175 | Train MSE: 26.490 | Validation MAE: 3.116
Epoch 200 | Train MSE: 25.943 | Validation MAE: 3.096
Epoch 225 | Train MSE: 25.378 | Validation MAE: 3.066
Epoch 250 | Train MSE: 24.561 | Validation MAE: 3.023
Epoch 275 | Train MSE: 23.838 | Validation MAE: 2.982
Epoch 300 | Train MSE: 23.251 | Validation MAE: 2.953
Epoch 325 | Train MSE: 22.703 | Validation MAE: 2.924
Epoch 350 | Train MSE: 22.089 | Validation MAE: 2.918
Epoch 375 | Train MSE: 21.447 | Validation MAE: 2.896
Epoch 400 | Train MSE: 20.986 | Validation MAE: 2.859
Epoch 425 | Train MSE: 20.527 | Validation MAE: 2.844
Epoch 450 | Train MSE: 20.210 | Validation MAE: 2.847
Epoch 475 | Train MSE: 19.924 |

In [12]:
def predict_gnn(df):
    batch = make_graph_batch(df).to(device)
    gnn.eval()
    with torch.no_grad():
        predictions = gnn(batch.x, batch.edge_index).cpu().numpy()
    # Batch rows are ordered by quarter, then the saved grid-cell node order.
    pred_index = pd.MultiIndex.from_product(
        [sorted(df["quarter"].unique()), cell_ids], names=["quarter", "cell_id"]
    )
    pred_lookup = pd.Series(predictions, index=pred_index)
    aligned = pred_lookup.reindex(pd.MultiIndex.from_frame(df[["quarter", "cell_id"]]))
    assert aligned.notna().all()
    return aligned.to_numpy()

val["model_d_pred"] = predict_gnn(val)
model_predictions = {
    "Model A — Naive": "baseline_pred",
    "Model B — Historical LightGBM": "model_b_pred",
    "Model C — Graph-enhanced LightGBM": "model_c_pred",
    "Model D — GCN": "model_d_pred"
}
comparison = pd.DataFrame({
    name: evaluate_model(val, col) for name, col in model_predictions.items()
}).T
print("2024 validation — primary metrics: Precision@20 and Capture@20")
display(comparison.round(3))

2024 validation — primary metrics: Precision@20 and Capture@20


,Precision@20,Capture@20,MAE,R²
Model A — Naive,0.550,0.189,2.757,0.766
Model B — Historical LightGBM,0.550,0.186,2.335,0.821
Model C — Graph-enhanced LightGBM,0.612,0.191,2.150,0.851
Model D — GCN,0.500,0.177,2.723,0.784


## 6. Exploratory evaluation on 2025

The 2025 results were inspected during development. These new evaluations are
**exploratory**, not an untouched final test. Model B and C hyperparameters are fixed
from 2024, then refitted on 2020–2024 as in the original pipeline. Model D uses its
2024-selected checkpoint trained on 2020–2023, with the training-only scaler; it is
not refitted here. This difference in fitting periods should accompany comparisons.
All 2025 predictions are recomputed from the current models.

In [13]:
train_final = pd.concat([train, val])

final_model_b = LGBMRegressor(
    **params_b,
    learning_rate=0.05,
    random_state=SEED,
    verbosity=-1,
    n_jobs=1
)

final_model_c = LGBMRegressor(
    **params_c,
    learning_rate=0.05,
    random_state=SEED,
    verbosity=-1,
    n_jobs=1
)

final_model_b.fit(
    train_final[historical_cols],
    train_final["collisions"]
)

final_model_c.fit(
    train_final[model_c_features],
    train_final["collisions"]
)

,learning_rate,0.05
,n_estimators,200
,min_child_samples,50
,random_state,42
,n_jobs,1
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None


In [14]:
test["baseline_pred"] = test["lag_1"]
test["model_b_pred"] = np.maximum(final_model_b.predict(test[historical_cols]), 0)
test["model_c_pred"] = np.maximum(final_model_c.predict(test[model_c_features]), 0)
test["model_d_pred"] = predict_gnn(test)

test_results_df = pd.DataFrame({
    name: evaluate_model(test, col) for name, col in model_predictions.items()
}).T
print("2025 exploratory evaluation — primary metrics: Precision@20 and Capture@20")
display(test_results_df.round(3))

2025 exploratory evaluation — primary metrics: Precision@20 and Capture@20


,Precision@20,Capture@20,MAE,R²
Model A — Naive,0.462,0.158,3.363,0.722
Model B — Historical LightGBM,0.512,0.163,2.799,0.797
Model C — Graph-enhanced LightGBM,0.562,0.168,2.801,0.796
Model D — GCN,0.488,0.156,3.353,0.724


In [15]:
# Keep the existing prediction filename and columns used by app.py; include Model D.
test[["cell_id", "quarter", "collisions", *model_predictions.values()]].to_csv(
    "../data/processed/test_predictions.csv", index=False
)

## 7. Save the current fitted models

LightGBM artifacts contain the selected models refitted on 2020–2024. The GCN
checkpoint contains the best 2024-validation state trained on 2020–2023, its eight
feature names, node order, adjacency, architecture, fitted scaler statistics, and
training metadata. Scaler arrays can be restored directly onto `StandardScaler`.
The CPU checkpoint can be loaded with `torch.load(..., map_location="cpu", weights_only=True)`.

In [16]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert final_model_b.feature_name_ == historical_cols
assert final_model_c.feature_name_ == model_c_features
assert gnn.conv1.in_channels == 8 and scaler.n_features_in_ == 8
assert np.isclose(evaluate_model(val, "model_d_pred")["MAE"], best_mae, atol=1e-4)

model_config = {
    "historical_cols": historical_cols,
    "graph_cols": graph_cols,
    "gnn_features": gnn_features,
    "target": "collisions",
    "model_b_params": params_b,
    "model_c_params": params_c,
    "seed": SEED,
    "lightgbm_fit_period": ["2020Q1", "2024Q4"],
    "gnn_fit_period": ["2020Q1", "2023Q4"],
    "validation_period": ["2024Q1", "2024Q4"],
    "evaluation_period": ["2025Q1", "2025Q4"],
    "evaluation_status": "exploratory"
}
# Lists and tensors keep this checkpoint portable with weights_only=True.
gnn_checkpoint = {
    "model_state_dict": {name: value.detach().cpu() for name, value in gnn.state_dict().items()},
    "architecture": {"num_features": 8, "hidden_channels": [32, 16]},
    "gnn_features": gnn_features,
    "cell_ids": cell_ids,
    "edge_index": edge_index.cpu(),
    "scaler": {
        "mean_": scaler.mean_.tolist(),
        "scale_": scaler.scale_.tolist(),
        "var_": scaler.var_.tolist(),
        "n_features_in_": int(scaler.n_features_in_),
        "n_samples_seen_": int(scaler.n_samples_seen_)
    },
    "best_epoch": best_epoch,
    "validation_mae": best_mae,
    "seed": SEED,
    "fit_period": ["2020Q1", "2023Q4"],
    "validation_period": ["2024Q1", "2024Q4"],
    "evaluation_status": "2025 exploratory"
}
joblib.dump(final_model_b, MODEL_DIR / "model_b.pkl")
joblib.dump(final_model_c, MODEL_DIR / "model_c.pkl")
joblib.dump(model_config, MODEL_DIR / "model_config.pkl")
torch.save(gnn_checkpoint, MODEL_DIR / "gnn_model.pth")
print("Saved the current 4-feature Model B, 8-feature Model C, and 8-feature GCN with scaler.")

Saved the current 4-feature Model B, 8-feature Model C, and 8-feature GCN with scaler.


In [17]:
# Verify persistence against the models and scaler fitted in this run.
from time import perf_counter
from threadpoolctl import threadpool_limits

verification_started = perf_counter()
# Limit already-initialized native thread pools as well as newly imported ones.
# This avoids the observed OpenMP stall in the CPU GCN verification step.
with threadpool_limits(limits=1):
    for filename, fitted_model, features in [
        ("model_b.pkl", final_model_b, historical_cols),
        ("model_c.pkl", final_model_c, model_c_features)
    ]:
        print(f"Reloading and checking {filename}...", flush=True)
        loaded = joblib.load(MODEL_DIR / filename)
        assert loaded.feature_name_ == features
        assert np.allclose(loaded.predict(test[features]), fitted_model.predict(test[features]))

    print("Reloading GCN checkpoint and checking scaler...", flush=True)
    checkpoint = torch.load(MODEL_DIR / "gnn_model.pth", map_location="cpu", weights_only=True)
    assert checkpoint["gnn_features"] == gnn_features
    assert checkpoint["cell_ids"] == cell_ids
    assert torch.equal(checkpoint["edge_index"], edge_index.cpu())
    restored_scaler = StandardScaler()
    for name in ["mean_", "scale_", "var_"]:
        setattr(restored_scaler, name, np.asarray(checkpoint["scaler"][name]))
    restored_scaler.n_features_in_ = checkpoint["scaler"]["n_features_in_"]
    restored_scaler.n_samples_seen_ = checkpoint["scaler"]["n_samples_seen_"]
    sample = train[gnn_features].to_numpy(dtype=np.float32)
    assert np.allclose(restored_scaler.transform(sample), scaler.transform(sample))

    print("Checking GCN predictions on CPU...", flush=True)
    loaded_gnn = RoadRiskGCN(checkpoint["architecture"]["num_features"])
    loaded_gnn.load_state_dict(checkpoint["model_state_dict"])
    loaded_gnn.eval()
    with torch.no_grad():
        restored_preds = loaded_gnn(val_batch.x.cpu(), val_batch.edge_index.cpu()).numpy()
    ordered_val = val.sort_values(["quarter", "cell_id"])
    assert np.allclose(restored_preds, ordered_val["model_d_pred"], rtol=1e-4, atol=1e-4)

print(f"Model and GNN scaler loading verified in {perf_counter() - verification_started:.2f}s.")


Reloading and checking model_b.pkl...
Reloading and checking model_c.pkl...
Reloading GCN checkpoint and checking scaler...
Checking GCN predictions on CPU...
Model and GNN scaler loading verified in 0.07s.


## 8. Summary

The executed comparison contains one result row for each of Models A–D. Model B
uses four historical features; Models C and D use the same eight inputs. The GCN
additionally uses shared-edge grid adjacency and a scaler fitted only on training rows.

In the recorded CPU run, Model C achieved the highest 2024 Precision@20 and
Capture@20; the GCN did not improve on it. Validation informed model selection,
and the previously inspected 2025 results remain exploratory. Saved artifacts and
predictions were regenerated for this feature configuration and verified by reloading.